Loading the Dataset and Extracting Features

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier  
from sklearn.metrics import accuracy_score, f1_score, precision_score,recall_score,confusion_matrix, classification_report

# Ensure the output directory exists
os.makedirs("confusion_matrices", exist_ok=True)

# File to store results
output_file = "results.txt"

# Load dataset
tup = pd.read_csv("test.csv")

# Convert dataset into tuple format
X_tuples = list(tup.itertuples(index=False, name=None))

# Extract features (all columns except the 4th one) and labels (4th column)
X = [list(t[:3]) + list(t[4:]) for t in X_tuples]  # Skip the 4th column
Y = [t[3] for t in X_tuples]  # 4th column as label

# Convert values to numeric, replacing NaN, inf, and -inf with 0
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
Y = np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0)





Splitting of Data into Training and Testing Data 

In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=0)


Standardization of the Dataset

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


Models Definition and Metrics Evaluation

In [13]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=0),
    "Decision Tree": DecisionTreeClassifier(random_state=0),
    "Gaussian Naïve Bayes": GaussianNB(),
    "SVM": SVC(kernel="linear", random_state=0),
    "RandomForestClassifier":RandomForestClassifier(n_estimators=100,random_state=0)
}
# Train and evaluate each model
results = {}
with open(output_file, "w") as f:
    for model_name, model in models.items():
        print(f"Training {model_name}...")
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)

        # Compute metrics
        accuracy = accuracy_score(Y_test, Y_pred)
        f1 = f1_score(Y_test, Y_pred, average='weighted')
        precision = precision_score(Y_test, Y_pred, average='weighted', zero_division=0)
        recall = recall_score(Y_test, Y_pred, average='weighted', zero_division=0)
        conf_matrix = confusion_matrix(Y_test, Y_pred)

        results[model_name] = {
            "Accuracy": accuracy,
            "F1 Score": f1,
            "Precision": precision,
            "Recall": recall,
            "Confusion Matrix": conf_matrix
        }

        # Save classification report to file
        f.write(f"\n===== {model_name} =====\n")
        f.write(f"Accuracy: {accuracy:.4f}\n")
        f.write(f"F1 Score: {f1:.4f}\n")
        f.write(f"Precision: {precision:.4f}\n")
        f.write(f"Recall: {recall:.4f}\n")
        f.write("Classification Report:\n")
        f.write(classification_report(Y_test, Y_pred, zero_division=0))
        f.write("\nConfusion Matrix:\n")
        np.savetxt(f, conf_matrix, fmt="%d")
        f.write("\n" + "-"*50 + "\n")

        # Plot and save confusion matrix
        plt.figure(figsize=(6, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=np.unique(Y), yticklabels=np.unique(Y))
        plt.title(f"Confusion Matrix - {model_name}")
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")

        # Save the figure
        plt.savefig(f"confusion_matrices/{model_name}_confusion_matrix.png")
        plt.close()  
print(f"\nResults saved in '{output_file}' and confusion matrices saved in 'confusion_matrices/' folder.")      


Training Logistic Regression...
Training Decision Tree...
Training Gaussian Naïve Bayes...
Training SVM...
Training RandomForestClassifier...

Results saved in 'results.txt' and confusion matrices saved in 'confusion_matrices/' folder.
